In [1]:
import os
import re
from pathlib import Path
import pandas as pd
from ollama import Client
from dotenv import load_dotenv

In [2]:
data_path = Path("../../data/processed/herd_mentality_events_processed_v2.csv").resolve()
processed_path = Path("../../data/processed/herd_mentality_events_processed_final.csv").resolve()

load_dotenv()
OLLAMA_API_KEY=os.getenv("OLLAMA_API_KEY")

In [3]:
df = pd.read_csv(data_path)
df

,Year,Event Name,Continent,Event Description,Country
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker...",South Africa
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le...",United Kingdom
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found...",South Africa
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S...",South Africa
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...,Mali
...,...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui...",Papua New Guinea
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic...",Fiji
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su...",Australia
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific...",Fiji


In [4]:
OLLAMA_API_BASE = "https://ollama.com"
OLLAMA_MODEL = "gpt-oss:120b"

# Initialize Ollama Cloud client with authentication
# NOTE: Verify your API key at https://ollama.com/settings/keys
# The API key should be a simple string, not an SSH key format
client = Client(host=OLLAMA_API_BASE, headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY})

def run_ollama_chat(prompt: str,
                    model: str = OLLAMA_MODEL,
                    temperature: float = 0.1,
                    max_tokens: int | None = 1000) -> str:
    """Send a chat prompt to Ollama Cloud and return the response text."""
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "num_predict": max_tokens if max_tokens else None
        }
    )
    # Ollama Client returns ChatResponse with message.content, not choices[0].message.content
    return response['message']['content']

result = run_ollama_chat("What is the capital of France?. Answer in 2 to 3 words.")
print(result)

Paris, France


In [5]:
def save_to_csv(df, path):
    df.to_csv(path, index=False)
    print(f"Data saved to {path}")

In [6]:
def calculate_decade(year):
    return str(year)[:3] + "0's"

In [7]:
df['Decade'] = df['Year'].apply(calculate_decade)

In [8]:
df

,Year,Event Name,Continent,Event Description,Country,Decade
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker...",South Africa,1920's
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le...",United Kingdom,1920's
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found...",South Africa,1920's
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S...",South Africa,1920's
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...,Mali,1930's
...,...,...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui...",Papua New Guinea,2020's
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic...",Fiji,2020's
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su...",Australia,2020's
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific...",Fiji,2020's


In [9]:
save_to_csv(df, processed_path)

Data saved to C:\RohitDir\Martian Data\data\processed\herd_mentality_events_processed_final.csv


## HBI INFORMATION AUGMENTATION

In [10]:
# HBI prompt components and validators
ALLOWED_EVENT_TYPES = {"Social", "Financial", "Consumer", "Technology", "Industrial"}
ALLOWED_TRIGGERS = {"Fear", "Greed", "Opportunity", "Repression", "Hype", "Policy"}

# Clamp scores to 1–10
clamp = lambda v: max(1, min(10, int(v))) if str(v).isdigit() else None

In [11]:
# Prompt builder for HBI (stable, deterministic output)
# Canonical system prompt for gpt-oss:120b — keep identical across uses
HBI_SYSTEM_PROMPT = """
You are an expert historical analyst constructing a Herd Behavior Index (HBI).
Follow all rules, definitions, and examples with strict consistency. If uncertain between two bins, choose the lower score. Use only the labels exactly as given.

============================================================
EVENT TYPE (choose ONE)
============================================================
Social | Financial | Consumer | Technology | Industrial

============================================================
TRIGGER (choose ONE)
============================================================
Fear | Greed | Opportunity | Repression | Hype | Policy

============================================================
HBI NUMERIC SCORES (1–10) WITH FULL LABEL DEFINITIONS
============================================================
Magnitude (M) — Size or scale of participation, economic value, or market affected
1 Minimal | 2 Very Low | 3 Low | 4 Moderate-Low | 5 Moderate | 6 Moderate-High | 7 High | 8 Very High | 9 Extreme | 10 Historic

Spread (S) — Geographic or demographic reach
1 Local | 2 Multi-Local | 3 City-Level | 4 Regional | 5 Multi-Regional | 6 National | 7 Bi-National | 8 Multi-National | 9 Continental | 10 Global

Intensity (I) — Severity or deviation from normal behavior
1 Minimal | 2 Very Low | 3 Low | 4 Moderate-Low | 5 Moderate | 6 Moderate-High | 7 High | 8 Very High | 9 Critical | 10 Maximum

Duration (D) — Time length of the event
1 Momentary (Hours) | 2 Very Short (1–2 days) | 3 Short (Several days) | 4 Moderate-Short (1–2 weeks) | 5 Moderate (Several weeks) | 6 Long (1–3 months) | 7 Extended (Several months) | 8 Very Long (6–12 months) | 9 Multi-Year (1–5 years) | 10 Prolonged (5+ years)

Outcome/Impact (R) — Long-term systemic or structural effect
1 Negligible | 2 Very Low | 3 Low | 4 Moderate-Low | 5 Moderate | 6 Strong | 7 Significant | 8 Major | 9 Transformational | 10 Historic

============================================================
FEW-SHOT EXAMPLES
============================================================
Example 1:
Event: "1925 South African Rand mine strike — Thousands of workers on the Witwatersrand launched a coordinated strike for better wages and conditions, resulting in violent clashes and arrests."
Event Type: Social | Trigger: Repression | Reason: large worker participation, regional reach, violent clashes, weeks-long, meaningful policy impact.
Scores → M: 8, S: 6, I: 7, D: 5, R: 7

Example 2:
Event: "1926 Pan-African Congress — African intellectuals and diaspora leaders organized a transcontinental anti-colonial platform."
Event Type: Social | Trigger: Policy | Reason: multi-continent participation, political significance, moderate intensity, short duration, major long-term influence.
Scores → M: 6, S: 9, I: 6, D: 4, R: 8

============================================================
INSTRUCTIONS FOR NEW EVENTS
============================================================
1) Read the event description carefully.
2) Choose ONE Event Type and ONE Trigger from the allowed lists; do not invent new labels.
3) Assign integers 1–10 for M, S, I, D, R using the label definitions above. If uncertain between two scores, pick the lower.
4) Output MUST start with the structured line exactly:
Event Type:<value>; Trigger:<value>; M:<int>; S:<int>; I:<int>; D:<int>; R:<int>
5) Optionally add a second line for human audit: Reason:<brief justification referencing the labels>. If you add it, keep it under 30 words.
"""

def build_hbi_prompt(row):
    return f"""
Event:
Year: {row['Year']}
Name: {row['Event Name']}
Continent: {row['Continent']}
Description: {row['Event Description']}

Respond with the required structured line (and an optional short reason line):
Event Type:<value>; Trigger:<value>; M:<int>; S:<int>; I:<int>; D:<int>; R:<int>
Reason:<brief justification>  # optional
"""

In [12]:
def parse_hbi_csv(text: str):
    """Parse the structured HBI line into a dict, enforcing allowed values and 1-10 ints.
    Only the first line is used (ignores optional Reason line). Accepts comma/semicolon separators.
    """
    first_line = text.strip().splitlines()[0]
    parts = [p.strip() for p in re.split(r"[;,]", first_line)]
    kv = {}
    for p in parts:
        if ':' not in p:
            continue
        k, v = p.split(':', 1)
        kv[k.strip().lower()] = v.strip()
    evt = kv.get('event type')
    trg = kv.get('trigger')
    M = clamp(kv.get('m'))
    S = clamp(kv.get('s'))
    I = clamp(kv.get('i'))
    D = clamp(kv.get('d'))
    R = clamp(kv.get('r'))
    if evt not in ALLOWED_EVENT_TYPES or trg not in ALLOWED_TRIGGERS:
        return None
    if None in [M, S, I, D, R]:
        return None
    return dict(event_type=evt, trigger=trg, M=M, S=S, I=I, D=D, R=R)

In [13]:
def infer_hbi(row, model=OLLAMA_MODEL, temperature=0.0):
    """Call Ollama client and parse HBI CSV line."""
    prompt = build_hbi_prompt(row)
    resp = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": HBI_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": temperature},
    )
    txt = resp['message']['content'].strip()
    parsed = parse_hbi_csv(txt)
    print(parsed)
    if not parsed:
        raise ValueError(f"Failed to parse HBI output: {txt}")
    return parsed

In [14]:
def enrich_with_hbi(df_slice: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for i, row in df_slice.iterrows():
        try:
            hbi = infer_hbi(row)
        except Exception as e:
            print(f"HBI error at row {i+1}: {e}")
            hbi = {"event_type": "Unknown", "trigger": "Unknown", "M": "", "S": "", "I": "", "D": "", "R": ""}
        out = row.to_dict()
        out.update({
            "Event Type": hbi["event_type"],
            "Trigger": hbi["trigger"],
            "M": hbi["M"],
            "S": hbi["S"],
            "I": hbi["I"],
            "D": hbi["D"],
            "R": hbi["R"],
        })
        rows.append(out)
    return pd.DataFrame(rows)

In [15]:
def process_hbi_batches(df: pd.DataFrame, batch_size: int = 100):
    from pathlib import Path
    import time

    batch_dir = processed_path.parent / "batches_hbi"
    batch_dir.mkdir(exist_ok=True)

    total = len(df)
    num_batches = (total + batch_size - 1) // batch_size
    all_files = []

    for b in range(num_batches):
        start = b * batch_size
        end = min(start + batch_size, total)
        batch_file = batch_dir / f"hbi_batch_{b+1:03d}.csv"

        if batch_file.exists():
            print(f"⏭ Batch {b+1}/{num_batches} exists, skipping")
            all_files.append(batch_file)
            continue

        print(f"Batch {b+1}/{num_batches} rows {start+1}-{end}...")
        batch_df = enrich_with_hbi(df.iloc[start:end])
        batch_df.to_csv(batch_file, index=False)
        all_files.append(batch_file)
        print(f"✓ Saved {batch_file.name}")
        if b < num_batches - 1:
            time.sleep(2)

    if len(all_files) == num_batches:
        print("Merging all batches...")
        merged = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
        merged.to_csv(processed_path, index=False)
        print(f"✓ Done! Final file: {processed_path}")
        return merged

    print(f"⚠ Not all batches complete ({len(all_files)}/{num_batches}). Re-run to resume.")
    return None

In [16]:
# Run HBI enrichment in batches (resume-friendly)
# Note: requires df already loaded (from earlier cell)
df_hbi = process_hbi_batches(df, batch_size=100)

⏭ Batch 1/7 exists, skipping
⏭ Batch 2/7 exists, skipping
⏭ Batch 3/7 exists, skipping
Batch 4/7 rows 301-400...
{'event_type': 'Social', 'trigger': 'Repression', 'M': 7, 'S': 6, 'I': 7, 'D': 5, 'R': 8}
{'event_type': 'Social', 'trigger': 'Repression', 'M': 7, 'S': 6, 'I': 8, 'D': 6, 'R': 7}
{'event_type': 'Social', 'trigger': 'Fear', 'M': 7, 'S': 6, 'I': 6, 'D': 5, 'R': 7}
{'event_type': 'Social', 'trigger': 'Policy', 'M': 6, 'S': 6, 'I': 7, 'D': 4, 'R': 7}
{'event_type': 'Social', 'trigger': 'Repression', 'M': 9, 'S': 6, 'I': 8, 'D': 7, 'R': 9}
{'event_type': 'Social', 'trigger': 'Fear', 'M': 7, 'S': 6, 'I': 6, 'D': 5, 'R': 8}
{'event_type': 'Social', 'trigger': 'Fear', 'M': 7, 'S': 6, 'I': 7, 'D': 4, 'R': 7}
{'event_type': 'Social', 'trigger': 'Fear', 'M': 7, 'S': 7, 'I': 6, 'D': 10, 'R': 9}
{'event_type': 'Social', 'trigger': 'Policy', 'M': 8, 'S': 6, 'I': 8, 'D': 8, 'R': 9}
{'event_type': 'Technology', 'trigger': 'Fear', 'M': 8, 'S': 9, 'I': 9, 'D': 5, 'R': 8}
{'event_type': 'Soci

In [24]:
def build_hbi_prompt():
    return f"""
Event:
Year: 1975
Name: Spanish transition: Franco’s death
Continent: Europe
Description: "On 20 November 1975 General Francisco Franco, Spain’s authoritarian ruler since the 1939 Civil War, died, ending a 36‑year dictatorship. His designated successor, King Juan Carlos I, was proclaimed head of state and promptly initiated a controlled opening toward democracy. The new monarch appointed reformist Prime Minister Adolfo Suárez, who oversaw the 1976 Law of Political Reform that dismantled the Francoist legal framework, legalized opposition parties—including the Communist Party—and scheduled free elections. The transition navigated entrenched military interests, regional nationalist demands, and a fragile economy, ultimately leading to Spain’s first democratic parliamentary elections in June 1977 and setting the foundation for the modern constitutional monarchy."

Respond with the required structured line (and an optional short reason line):
Event Type:<value>; Trigger:<value>; M:<int>; S:<int>; I:<int>; D:<int>; R:<int>
Reason:<brief justification>  # optional
"""

prompt = build_hbi_prompt()

model = OLLAMA_MODEL
temperature = 0.0

resp = client.chat(
    model=model,
    messages=[
        {"role": "system", "content": HBI_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ],
    options={"temperature": temperature},
)
txt = resp['message']['content'].strip()
parsed = parse_hbi_csv(txt)
print(parsed)

{'event_type': 'Social', 'trigger': 'Opportunity', 'M': 8, 'S': 6, 'I': 6, 'D': 9, 'R': 9}
